# Choice of the pivot

In [ ]:
#    APM41012EP course notebook - Chapter 4 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Analysis of the stability of direct algorithms for solving linear systems
#         Pivoting or not, partial or total pivoting.
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from mpmath import mp

from scipy.stats import ortho_group

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def gaussian_elimination_without_pivoting(a, b): 
    n = b.size
    for i in range(n-1):
        # elimination
        li = a[i+1:,i]/a[i,i]
        b[i+1:] = b[i+1:] - li * b[i]
        a[i+1:] = a[i+1:] - li.reshape(n-i-1,1)*a[i]

def gaussian_elimination_with_partial_pivoting(a, b): 
    n = b.size
    for i in range(n-1):
        # partial pivoting    
        i_max = np.argmax(np.abs(a[i:,i]))
        a[[i,i_max+i]] = a[[i_max+i,i]]
        b[[i,i_max+i]] = b[[i_max+i,i]]
        # elimination
        li = a[i+1:,i]/a[i,i]
        b[i+1:] = b[i+1:] - li * b[i]
        a[i+1:] = a[i+1:] - li.reshape(n-i-1,1)*a[i]
        
def gaussian_elimination_with_total_pivoting(a, b): 

    n = b.size
    ord_col = np.arange(n)
    for i in range(n-1):
        # total pivoting
        i_max, j_max = np.unravel_index(np.abs(a[i:,i:]).argmax(), a[i:,i:].shape)
        a[[i,i_max+i]] = a[[i_max+i,i]]
        a[:,[i,j_max+i]] = a[:,[j_max+i,i]]
        b[[i,i_max+i]] = b[[i_max+i,i]]
        ord_col[[i,j_max+i]] = ord_col[[j_max+i,i]]
        # elimination
        li = a[i+1:,i]/a[i,i]
        b[i+1:] = b[i+1:] - li * b[i]
        a[i+1:] = a[i+1:] - li.reshape(n-i-1,1)*a[i]
        
    return ord_col

def backward_substitution(a, b):
    n = b.size
    x = np.empty(n)
    for i in range(n-1, -1, -1):
        x[i] = (b[i] - np.sum(a[i,i+1:]*x[i+1:])) / a[i,i]
    return x

def gauss_solve_without_pivoting(a, b):
    ag = np.copy(a) 
    bg = np.copy(b)
    gaussian_elimination_without_pivoting(ag, bg)
    return backward_substitution(ag, bg)

def gauss_solve_with_partial_pivoting(a, b):
    ag = np.copy(a) 
    bg = np.copy(b)
    gaussian_elimination_with_partial_pivoting(ag, bg)
    return backward_substitution(ag, bg)


def gauss_solve_with_total_pivoting(a, b):
    ag = np.copy(a) 
    bg = np.copy(b)
    ord_col = gaussian_elimination_with_total_pivoting(ag, bg)
    x = backward_substitution(ag, bg)
    return x[ord_col.argsort()]

## Random matrices

For each size n = 5,6,...,55 we choose 80 random matrices with coefficients $a_{ij}$ uniformly distributed in $[-1, 1]$ and solutions $x_i$ uniformly distributed in $[-1, 1]$ in double precision. We then compute in quadruple precision the $b_j$ for this exact solution. Next we apply the Gaussian algorithm, once without pivoting, then with partial pivoting and with total pivoting, in double precision. 

Having access to both the exact solution and a double-precision solution for the various algorithms allows an evaluation of the error generated by the use of the algorithm in double precision.

In [ ]:
def test_gauss_without_pivoting(n_test):

    err = np.zeros((51,n_test))
    errovercond = np.zeros((51,n_test))

    print("\nGauss without pivoting")

    for i, i_n in enumerate(range(5,56)):
        for j in range(n_test):
            a = mp.mpf('2.0')*np.random.random((i_n,i_n))-1
            # uncomment to see the impact of total pivoting
            # a[1,i_n-1] = mp.mpf('100000')
            x_ex = mp.mpf('2.0')*np.random.random(i_n)-1
            b = np.dot(a, x_ex)
            a_64 = a.astype(np.float64)
            b_64 = b.astype(np.float64)
            x_num = gauss_solve_without_pivoting(a_64, b_64)
            err[i,j] = np.linalg.norm(np.abs(x_num-x_ex), np.inf)
            errovercond[i,j] = err[i,j] / np.linalg.cond(a_64, np.inf)
            
    return err, errovercond

def test_gauss_with_partial_pivoting(n_test):

    err = np.zeros((51,n_test))
    errovercond = np.zeros((51,n_test))
        
    print("\nGauss with partial pivoting")
 
    for i, i_n in enumerate(range(5,56)):
        #print(".", end="")
        for j in range(n_test):
            a = mp.mpf('2.0')*np.random.random((i_n,i_n))-1
            # uncomment to see the impact of total pivoting
            # a[1,i_n-1] = mp.mpf('100000')
            x_ex = mp.mpf('2.0')*np.random.random(i_n)-1
            b = np.dot(a, x_ex)
            a_64 = a.astype(np.float64)
            b_64 = b.astype(np.float64)
            x_num = gauss_solve_with_partial_pivoting(a_64, b_64)
            err[i,j] = np.linalg.norm(np.abs(x_num-x_ex), np.inf)
            errovercond[i,j] = err[i,j] / np.linalg.cond(a_64, np.inf)
            
    return err, errovercond

def test_gauss_with_total_pivoting(n_test):

    err = np.zeros((51,n_test))
    errovercond = np.zeros((51,n_test))
        
    print("\nGauss with total pivoting")
 
    for i, i_n in enumerate(range(5,56)):
        for j in range(n_test):
            a = mp.mpf('2.0')*np.random.random((i_n,i_n))-1
            # uncomment to see the impact of total pivoting
            # a[1,i_n-1] = mp.mpf('100000')
            x_ex = mp.mpf('2.0')*np.random.random(i_n)-1
            b = np.dot(a, x_ex)
            a_64 = a.astype(np.float64)
            b_64 = b.astype(np.float64)
            x_num = gauss_solve_with_total_pivoting(a_64, b_64)
            err[i,j] = np.linalg.norm(np.abs(x_num-x_ex), np.inf)
            errovercond[i,j] = err[i,j] / np.linalg.cond(a_64, np.inf)
            
    return err, errovercond

In [ ]:
# number of matrices tested per size
n_test = 80

err_1, errovercond_1 = test_gauss_without_pivoting(n_test)
err_2, errovercond_2 = test_gauss_with_partial_pivoting(n_test)
err_3, errovercond_3 = test_gauss_with_total_pivoting(n_test)

The infinity norm of the error $(x_i^{num} - x_i^{ex})$, then the infinity norm of the error divided by the conditioning of each result, is shown in the following figure:

In [ ]:
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.1, subplot_titles=("|err|_inf", "|err|_inf", "|err|_inf", "|err|_inf/cond", "|err|_inf/cond", "|err|_inf/cond"))

for i, i_n in enumerate(range(5,56)):
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=err_1[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=1, col=1)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=err_2[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=1, col=2)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=err_3[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=1, col=3)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=errovercond_1[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=2, col=1)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=errovercond_2[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=2, col=2)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=errovercond_3[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=2, col=3)

fig.update_yaxes(type="log", range=[-17,-8],  exponentformat = 'e', row=1)    
fig.update_yaxes(type="log", range=[-19,-11], exponentformat = 'e', row=2)    
    
fig.update_layout(height=1000)

fig.add_annotation(text="Gauss without pivoting", font=dict(size=16), xanchor="center", xref="paper", yref="paper", showarrow=False, x=0.15, y=1.1)
fig.add_annotation(text="Gauss with partial pivoting", font=dict(size=16), xanchor="center", xref="paper", yref="paper", showarrow=False, x=0.5, y=1.1)
fig.add_annotation(text="Gauss with total pivoting", font=dict(size=16), xanchor="center", xref="paper", yref="paper", showarrow=False, x=0.85, y=1.1)

fig.show()

In the case without pivoting, the general error level is much higher than in the case with pivoting. The conditioning is the same in both cases, that of
the matrix $A$.

In order to pinpoint the origin of the errors, we have plotted the previous error divided by the conditioning of the matrix. It clearly appears that this number stays below the machine precision in double precision when a pivoting technique is used. This indicates an algorithm that is very stable in the backward sense. This is not the case for the algorithm without pivoting, which can cause serious accuracy problems. The choice of partial pivoting leads to an algorithm that behaves well in most cases but at a lower cost than total pivoting, which is used more rarely.

## Orthogonal matrices

For this second numerical experiment, we draw 80 random orthogonal matrices $a_{ij}$. This matrix is computed in double precision, the rest of the experiment continues as before.

In [ ]:
def test_ortho_gauss_without_pivoting(n_test):

    err = np.zeros((51,n_test))
    errovercond = np.zeros((51,n_test))

    print("\nGauss without pivoting")

    for i, i_n in enumerate(range(5,56)):
        for j in range(n_test):
            a = mp.mpf('1.0')*ortho_group.rvs(dim=i_n)
            x_ex = mp.mpf('2.0')*np.random.random(i_n)-1
            b = np.dot(a, x_ex)
            a_64 = a.astype(np.float64)
            b_64 = b.astype(np.float64)
            x_num = gauss_solve_without_pivoting(a_64, b_64)
            err[i,j] = np.linalg.norm(np.abs(x_num-x_ex), np.inf)
            errovercond[i,j] = err[i,j] / np.linalg.cond(a_64, np.inf)
            
    return err, errovercond

def test_ortho_gauss_with_partial_pivoting(n_test):

    err = np.zeros((51,n_test))
    errovercond = np.zeros((51,n_test))
        
    print("\nGauss with partial pivoting")
 
    for i, i_n in enumerate(range(5,56)):
        for j in range(n_test):
            a = mp.mpf('1.0')*ortho_group.rvs(dim=i_n)
            x_ex = 2*np.random.random(i_n)-1
            b = np.dot(a, x_ex)
            a_64 = a.astype(np.float64)
            b_64 = b.astype(np.float64)
            x_num = gauss_solve_with_partial_pivoting(a_64, b_64)
            err[i,j] = np.linalg.norm(np.abs(x_num-x_ex), np.inf)
            errovercond[i,j] = err[i,j] / np.linalg.cond(a_64, np.inf)
            
    return err, errovercond

def test_ortho_gauss_with_total_pivoting(n_test):

    err = np.zeros((51,n_test))
    errovercond = np.zeros((51,n_test))
        
    print("\nGauss with total pivoting")
 
    for i, i_n in enumerate(range(5,56)):
        for j in range(n_test):
            a = mp.mpf('1.0')*ortho_group.rvs(dim=i_n)
            x_ex = 2*np.random.random(i_n)-1
            b = np.dot(a, x_ex)
            a_64 = a.astype(np.float64)
            b_64 = b.astype(np.float64)
            x_num = gauss_solve_with_total_pivoting(a_64, b_64)
            err[i,j] = np.linalg.norm(np.abs(x_num-x_ex), np.inf)
            errovercond[i,j] = err[i,j] / np.linalg.cond(a_64, np.inf)
            
    return err, errovercond

In [ ]:
# number of matrices tested per size
n_test = 80

err_4, errovercond_4 = test_ortho_gauss_without_pivoting(n_test)
err_5, errovercond_5 = test_ortho_gauss_with_partial_pivoting(n_test)
err_6, errovercond_6 = test_ortho_gauss_with_total_pivoting(n_test)

The infinity norm of the error $(x_i^{num} - x_i^{ex})$, then the infinity norm of the error divided by the conditioning of each result, is shown in the following figure:

In [ ]:
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.1, subplot_titles=("|err|_inf", "|err|_inf", "|err|_inf", "|err|_inf/cond", "|err|_inf/cond", "|err|_inf/cond>"))

for i, i_n in enumerate(range(5,56)):
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=err_4[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=1, col=1)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=err_5[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=1, col=2)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=err_6[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=1, col=3)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=errovercond_4[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=2, col=1)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=errovercond_5[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=2, col=2)
    fig.add_trace(go.Scatter(x=np.ones(n_test)*i_n, y=errovercond_6[i], showlegend=False, mode="markers", marker_color='blue', marker_size=3), row=2, col=3)

fig.update_yaxes(type="log", range=[-17,-8],  exponentformat = 'e', row=1)    
fig.update_yaxes(type="log", range=[-18,-10], exponentformat = 'e', row=2)

fig.add_annotation(text="Gauss without pivoting", font=dict(size=16), xanchor="center", xref="paper", yref="paper", showarrow=False, x=0.15, y=1.1)
fig.add_annotation(text="Gauss with partial pivoting", font=dict(size=16), xanchor="center", xref="paper", yref="paper", showarrow=False, x=0.5, y=1.1)
fig.add_annotation(text="Gauss with total pivoting", font=dict(size=16), xanchor="center", xref="paper", yref="paper", showarrow=False, x=0.85, y=1.1)
    
fig.update_layout(height=1000)

fig.show()

We observe that when we use partial pivoting, there is no exception to the good performance of the Gaussian algorithm, since one cannot have an ill-conditioned matrix in this case!

Moreover, the conclusions are the same regarding the stability of the algorithms, except that the errors here are generated solely by the potential loss of stability of the algorithms, since all the matrices are well conditioned.

Note that it is the errors in the infinity norm that are presented, and that a slight growth with the dimension of the matrices is observed, which would not be the case in the 2-norm.